# Import

In [1]:
# %pip install ruwordnet pymorphy3 razdel

In [2]:
# ! ruwordnet download

In [3]:
import re


from ruwordnet import RuWordNet
from razdel import tokenize
from pymorphy3 import MorphAnalyzer

# Consts

In [4]:
RUS_STOPWORDS = {
    "и", "в", "во", "на", "с", "со", "а", "но", "или", "что", "как",
    "к", "ко", "из", "за", "по", "под", "у", "о", "об", "от", "до",
    "не", "ни", "же", "ли", "бы", "я", "ты", "он", "она", "оно", "мы",
    "вы", "они", "это", "тот", "та", "те", "этот", "эта", "эти",
    "буду", "будет", "будут", "был", "была", "были", "есть"
}

In [5]:
wn = RuWordNet()
morph = MorphAnalyzer()

In [6]:
class Sense:
    def __init__(self, sense_id, lemmas, definition, examples=None, hypernyms=None, hyponyms=None):
        self.sense_id = sense_id
        self.lemmas = lemmas
        self.definition = definition
        self.examples = examples or []
        self.hypernyms = hypernyms or []
        self.hyponyms = hyponyms or []

    def all_synonyms(self):
        return self.lemmas

# Tokenize and normalize

In [7]:
def is_russian_word(token):
    return bool(re.fullmatch(r"[а-яёА-ЯЁ-]+", token))

def lemmatize_word(word):
    parsed = morph.parse(word)[0]
    return parsed.normal_form

def tokenize_and_lemmatize(text, remove_stopwords=True):
    lemmas = []
    for tok in tokenize(text):
        word = tok.text.lower()
        if not is_russian_word(word):
            continue
        lemma = lemmatize_word(word)
        if remove_stopwords and lemma in RUS_STOPWORDS:
            continue
        lemmas.append(lemma)
    return lemmas

def get_context(sentence, target_word, remove_target=True):
    sentence_lemmas = tokenize_and_lemmatize(sentence)
    target_lemma = lemmatize_word(target_word.lower())
    
    if remove_target:
        sentence_lemmas = [w for w in sentence_lemmas if w != target_lemma]
    
    return sentence_lemmas

# Find senses in RuWordNet

In [8]:
class RussianThesaurus:
    def __init__(self, wn):
        self.wn = wn

    def _safe_text(self, value):
        if value is None:
            return ""
        return str(value).strip()

    def _extract_definition(self, synset):
        # В разных версиях/объектах поле может называться по-разному
        for attr in ("definition", "description", "text", "title"):
            if hasattr(synset, attr):
                val = getattr(synset, attr)
                if val:
                    return self._safe_text(val)
        return self._safe_text(getattr(synset, "title", ""))

    def _extract_examples(self, synset):
        if hasattr(synset, "examples"):
            ex = getattr(synset, "examples")
            if ex:
                return [self._safe_text(x) for x in ex if self._safe_text(x)]
        return []

    def _extract_synonyms(self, synset):
        synonyms = []
        if hasattr(synset, "senses"):
            for s in synset.senses:
                name = getattr(s, "name", None)
                if name:
                    synonyms.append(self._safe_text(name).lower())
      
        seen = set()
        result = []
        for item in synonyms:
            if item not in seen:
                seen.add(item)
                result.append(item)
        return result

    def _extract_related_titles(self, synsets):
        result = []
        for syn in synsets:
            if hasattr(syn, "title") and syn.title:
                result.append(self._safe_text(syn.title).lower())
            elif hasattr(syn, "senses"):
                names = [self._safe_text(s.name).lower() for s in syn.senses if getattr(s, "name", None)]
                if names:
                    result.append(", ".join(names))

        seen = set()
        uniq = []
        for item in result:
            if item not in seen:
                seen.add(item)
                uniq.append(item)
        return uniq

    def get_senses(self, word):
        lemma = lemmatize_word(word.lower())
        raw_senses = self.wn.get_senses(lemma)

        results = []
        for raw_sense in raw_senses:
            synset = raw_sense.synset

            sense_id = self._safe_text(getattr(raw_sense, "id", "")) or self._safe_text(getattr(synset, "id", ""))
            lemmas = self._extract_synonyms(synset)

            if not lemmas:
                raw_name = getattr(raw_sense, "name", None)
                if raw_name:
                    lemmas = [self._safe_text(raw_name).lower()]
                else:
                    lemmas = [lemma]

            definition = self._extract_definition(synset)
            examples = self._extract_examples(synset)

            hypernyms = self._extract_related_titles(getattr(synset, "hypernyms", []))
            hyponyms = self._extract_related_titles(getattr(synset, "hyponyms", []))

            results.append(
                Sense(
                    sense_id=sense_id,
                    lemmas=lemmas,
                    definition=definition,
                    examples=examples,
                    hypernyms=hypernyms,
                    hyponyms=hyponyms,
                )
            )

        return results

# Lesk alg

In [9]:
def build_signature(sense):
    signature_text_parts = []

    # леммы / синонимы значения
    signature_text_parts.extend(sense.lemmas)

    # определение
    signature_text_parts.append(sense.definition)

    # примеры
    signature_text_parts.extend(sense.examples)

    # связанные понятия
    signature_text_parts.extend(sense.hypernyms)
    signature_text_parts.extend(sense.hyponyms)

    signature_text = " ".join([part for part in signature_text_parts if part])
    signature_lemmas = set(tokenize_and_lemmatize(signature_text))
    return signature_lemmas


def compute_overlap(context_words, signature_words):
    return set(context_words) & signature_words


def build_signature_parts(sense):
    return {
        "lemmas": set(tokenize_and_lemmatize(" ".join(sense.lemmas))),
        "definition_examples": set(tokenize_and_lemmatize(" ".join([sense.definition, *sense.examples]))),
        "hypernyms": set(tokenize_and_lemmatize(" ".join(sense.hypernyms))),
        "hyponyms": set(tokenize_and_lemmatize(" ".join(sense.hyponyms))),
    }


def compute_weighted_overlap(
    context_words,
    signature_parts,
    weights,
):
    context_set = set(context_words)
    total_score = 0.0
    overlap_by_part = {}
    total_overlap = set()

    for part_name, words in signature_parts.items():
        overlap = context_set & words
        overlap_by_part[part_name] = overlap
        total_overlap.update(overlap)
        total_score += len(overlap) * weights.get(part_name, 0.0)

    return total_score, overlap_by_part, total_overlap


def simple_lesk(sentence, target_word, thesaurus):
    senses = thesaurus.get_senses(target_word)
    if not senses:
        raise ValueError(f"Для слова '{target_word}' не найдено значений в тезаурусе.")

    context = get_context(sentence, target_word)
    context_set = set(context)

    best_sense = None
    best_score = -1
    best_overlap = set()
    ranking = []

    for sense in senses:
        signature = build_signature(sense)
        overlap = compute_overlap(context, signature)
        score = len(overlap)

        ranking.append({
            "sense_id": sense.sense_id,
            "lemmas": sense.lemmas,
            "definition": sense.definition,
            "score": score,
            "overlap": sorted(list(overlap)),
            "signature": sorted(list(signature))
        })

        if score > best_score:
            best_score = score
            best_sense = sense
            best_overlap = overlap

    all_zero = all(item["score"] == 0 for item in ranking)
    winners = [item for item in ranking if item["score"] == best_score]
    is_tie = len(winners) > 1

    return {
        "target_word": target_word,
        "target_lemma": lemmatize_word(target_word.lower()),
        "context": sorted(list(context_set)),
        "best_sense": best_sense,
        "best_score": best_score,
        "best_overlap": sorted(list(best_overlap)),
        "ranking": ranking,
        "all_zero": all_zero,
        "is_tie": is_tie
    }


def updated_lesk(sentence, target_word, thesaurus, weights=None):
    if weights is None:
        weights = {
            "lemmas": 3.0,
            "definition_examples": 2.0,
            "hypernyms": 1.0,
            "hyponyms": 0.5,
        }

    senses = thesaurus.get_senses(target_word)
    if not senses:
        raise ValueError(f"Для слова '{target_word}' не найдено значений в тезаурусе.")

    context = get_context(sentence, target_word)
    context_set = set(context)

    best_sense = None
    best_score = -1.0
    best_overlap = set()
    ranking = []

    for sense in senses:
        signature_parts = build_signature_parts(sense)
        score, overlap_by_part, total_overlap = compute_weighted_overlap(context, signature_parts, weights)

        ranking.append({
            "sense_id": sense.sense_id,
            "lemmas": sense.lemmas,
            "definition": sense.definition,
            "score": score,
            "overlap": sorted(list(total_overlap)),
            "overlap_by_part": {
                part_name: sorted(list(words))
                for part_name, words in overlap_by_part.items()
                if words
            },
            "score_by_part": {
                part_name: len(words) * weights.get(part_name, 0.0)
                for part_name, words in overlap_by_part.items()
                if words
            },
            "signature": sorted(list(set().union(*signature_parts.values()))),
            "signature_parts": {
                part_name: sorted(list(words))
                for part_name, words in signature_parts.items()
            },
        })

        if score > best_score:
            best_score = score
            best_sense = sense
            best_overlap = total_overlap

    all_zero = all(item["score"] == 0 for item in ranking)
    winners = [item for item in ranking if item["score"] == best_score]
    is_tie = len(winners) > 1

    return {
        "target_word": target_word,
        "target_lemma": lemmatize_word(target_word.lower()),
        "context": sorted(list(context_set)),
        "best_sense": best_sense,
        "best_score": best_score,
        "best_overlap": sorted(list(best_overlap)),
        "ranking": ranking,
        "all_zero": all_zero,
        "is_tie": is_tie,
        "weights": weights,
    }

# Output

In [10]:
def print_wsd_result(result):
    best_sense = result["best_sense"]

    print("=" * 80)
    print(f"Целевое слово: {result['target_word']}")
    print(f"Лемма целевого слова: {result['target_lemma']}")
    print(f"Контекст: {', '.join(result['context'])}")
    print("=" * 80)

    if result.get("all_zero"):
        print("ВНИМАНИЕ: у всех значений score = 0.")
        print("Буквальных пересечений между контекстом и сигнатурами не найдено.")
        print("Выбор значения в этом случае ненадёжен.")
        print("=" * 80)

    elif result.get("is_tie"):
        print("ВНИМАНИЕ: несколько значений имеют одинаковый максимальный score.")
        print("Выбор значения может быть неоднозначным.")
        print("=" * 80)

    print("Ранжирование значений:")
    for i, item in enumerate(result["ranking"], start=1):
        print(f"\n[{i}] sense_id = {item['sense_id']}")
        print(f"Леммы: {', '.join(item['lemmas']) if item['lemmas'] else '—'}")
        print(f"Определение: {item['definition'] if item['definition'] else '—'}")
        print(f"Score: {item['score']}")
        if item.get("score_by_part"):
            score_breakdown = ", ".join(
                [f"{part}={value:g}" for part, value in item["score_by_part"].items()]
            )
            print(f"Вклад по частям: {score_breakdown}")
        if item.get("overlap_by_part"):
            overlap_breakdown = "; ".join(
                [f"{part}: {', '.join(words)}" for part, words in item["overlap_by_part"].items()]
            )
            print(f"Пересечение по частям: {overlap_breakdown}")
        print(f"Пересечение: {', '.join(item['overlap']) if item['overlap'] else '—'}")

    print("\n" + "=" * 80)
    print("ВЫБРАННОЕ ЗНАЧЕНИЕ")
    print(f"ID: {best_sense.sense_id}")
    print(f"Синонимы: {', '.join(best_sense.all_synonyms()) if best_sense.all_synonyms() else '—'}")
    print(f"Определение: {best_sense.definition if best_sense.definition else '—'}")
    print(f"Гиперонимы: {', '.join(best_sense.hypernyms) if best_sense.hypernyms else '—'}")
    print(f"Гипонимы: {', '.join(best_sense.hyponyms) if best_sense.hyponyms else '—'}")
    print(f"Совпавшие слова: {', '.join(result['best_overlap']) if result['best_overlap'] else '—'}")
    print("=" * 80)

# Result

In [11]:
thesaurus = RussianThesaurus(wn)

sentence = "Я буду сдавать лабораторные работы  в университете вовремя и получу хорошую оценку за экзамен и буду хорошо учиться."
target_word = "оценку"

result = simple_lesk(sentence, target_word, thesaurus)
print_wsd_result(result)

Целевое слово: оценку
Лемма целевого слова: оценка
Контекст: быть, вовремя, лабораторный, получить, работа, сдавать, университет, учиться, хороший, хорошо, экзамен
ВНИМАНИЕ: несколько значений имеют одинаковый максимальный score.
Выбор значения может быть неоднозначным.
Ранжирование значений:

[1] sense_id = 113429-N-145932
Леммы: отметка, оценка, учебная отметка, оценка ученика, оценка учащегося, учебная оценка
Определение: отметка, выставляемая в учебных заведениях
Score: 1
Пересечение: учиться

[2] sense_id = 106775-N-145932
Леммы: оценка, оценочная характеристика
Определение: мнение, суждение или отзыв о чём-либо
Score: 1
Пересечение: хороший

[3] sense_id = 115032-N-145932
Леммы: оценка, оценка стоимости, оценивание, оценивание стоимости
Определение: определить, установить стоимость, цену, ценность чего-либо
Score: 0
Пересечение: —

[4] sense_id = 115033-N-145932
Леммы: аттестация, оценка, оценивание
Определение: мнение, суждение или отзыв о чём-либо
Score: 0
Пересечение: —

ВЫБРА

In [12]:
thesaurus = RussianThesaurus(wn)

sentence = "Я буду сдавать лабораторные работы вовремя и получу хорошую оценку за экзамен, надеюсь, пятёрку или четвёрку"
target_word = "оценку"

result = simple_lesk(sentence, target_word, thesaurus)
print_wsd_result(result)

print("\nПРОВЕРКА СИГНАТУР:")
for i, item in enumerate(result["ranking"], start=1):
    print(f"\n[{i}] {item['sense_id']}")
    print("Сигнатура:", ", ".join(item["signature"]))

Целевое слово: оценку
Лемма целевого слова: оценка
Контекст: быть, вовремя, лабораторный, надеяться, получить, пятёрка, работа, сдавать, хороший, четвёрка, экзамен
Ранжирование значений:

[1] sense_id = 113429-N-145932
Леммы: отметка, оценка, учебная отметка, оценка ученика, оценка учащегося, учебная оценка
Определение: отметка, выставляемая в учебных заведениях
Score: 2
Пересечение: пятёрка, четвёрка

[2] sense_id = 106775-N-145932
Леммы: оценка, оценочная характеристика
Определение: мнение, суждение или отзыв о чём-либо
Score: 1
Пересечение: хороший

[3] sense_id = 115032-N-145932
Леммы: оценка, оценка стоимости, оценивание, оценивание стоимости
Определение: определить, установить стоимость, цену, ценность чего-либо
Score: 0
Пересечение: —

[4] sense_id = 115033-N-145932
Леммы: аттестация, оценка, оценивание
Определение: мнение, суждение или отзыв о чём-либо
Score: 0
Пересечение: —

ВЫБРАННОЕ ЗНАЧЕНИЕ
ID: 113429-N-145932
Синонимы: отметка, оценка, учебная отметка, оценка ученика, оце

In [13]:
thesaurus = RussianThesaurus(wn)

sentence = "Я буду сдавать лабораторные работы вовремя и получу хорошую оценку за экзамен, надеюсь, пятёрку или четвёрку"
target_word = "оценку"

result = updated_lesk(sentence, target_word, thesaurus)
print_wsd_result(result)

print("\nПРОВЕРКА СИГНАТУР:")
for i, item in enumerate(result["ranking"], start=1):
    print(f"\n[{i}] {item['sense_id']}")
    print("Сигнатура:", ", ".join(item["signature"]))

Целевое слово: оценку
Лемма целевого слова: оценка
Контекст: быть, вовремя, лабораторный, надеяться, получить, пятёрка, работа, сдавать, хороший, четвёрка, экзамен
Ранжирование значений:

[1] sense_id = 113429-N-145932
Леммы: отметка, оценка, учебная отметка, оценка ученика, оценка учащегося, учебная оценка
Определение: отметка, выставляемая в учебных заведениях
Score: 1.0
Вклад по частям: hyponyms=1
Пересечение по частям: hyponyms: пятёрка, четвёрка
Пересечение: пятёрка, четвёрка

[2] sense_id = 106775-N-145932
Леммы: оценка, оценочная характеристика
Определение: мнение, суждение или отзыв о чём-либо
Score: 0.5
Вклад по частям: hyponyms=0.5
Пересечение по частям: hyponyms: хороший
Пересечение: хороший

[3] sense_id = 115032-N-145932
Леммы: оценка, оценка стоимости, оценивание, оценивание стоимости
Определение: определить, установить стоимость, цену, ценность чего-либо
Score: 0.0
Пересечение: —

[4] sense_id = 115033-N-145932
Леммы: аттестация, оценка, оценивание
Определение: мнение, с

# Try

In [14]:
thesaurus = RussianThesaurus(wn)

sentence = input("Введите предложение для анализа: ")
target_word = input("Введите слово для анализа (из предложения): ")

result = updated_lesk(sentence, target_word, thesaurus)
print_wsd_result(result)

Целевое слово: лук
Лемма целевого слова: лук
Контекст: выращивать, любимый, любить, мой, растение
Ранжирование значений:

[1] sense_id = 144182-N-106039
Леммы: лук, лучок
Определение: ЛУК (РАСТЕНИЕ)
Score: 3.0
Вклад по частям: definition_examples=2, hypernyms=1
Пересечение по частям: definition_examples: растение; hypernyms: растение
Пересечение: растение

[2] sense_id = 144347-N-106039
Леммы: лук, зеленый лук, лук-перо, лук зеленый, перо лука, лучок
Определение: ЗЕЛЕНЫЙ ЛУК
Score: 1.0
Вклад по частям: hypernyms=1
Пересечение по частям: hypernyms: растение
Пересечение: растение

[3] sense_id = 108010-N-106039
Леммы: лук
Определение: ЛУК (ОРУЖИЕ)
Score: 0.0
Пересечение: —

ВЫБРАННОЕ ЗНАЧЕНИЕ
ID: 144182-N-106039
Синонимы: лук, лучок
Определение: ЛУК (РАСТЕНИЕ)
Гиперонимы: луковичное растение, овощи, овощная культура
Гипонимы: лук-порей, репчатый лук, лук-шалот, зеленый лук, черемша
Совпавшие слова: растение
